In [2]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Masking, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [3]:
# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
DB_NAME = "../../nba_data.db"
DB_URI = f"sqlite:///{DB_NAME}"
engine = create_engine(DB_URI, echo=False)

In [4]:
# ------------------------------------------------------------
# Load Data & Sort
# ------------------------------------------------------------
df = pd.read_sql("SELECT * FROM player_game_features", engine)

# Ensure data is sorted by player and date
df = df.sort_values(by=["player_id", "game_date"])

# Extract the year from 'game_date'
df['game_year'] = pd.to_datetime(df['game_date']).dt.year

# Features and target
features = ["player_id", "pts", "min", "fgm", "fga", "pts_per_min", "fg_pct"]
target = "pts"

df = df.dropna(subset=features + ["pts"])

In [5]:
# ------------------------------------------------------------
# Helper Functions
# ------------------------------------------------------------
# NOTE: Data is the scaled dataframe, target is the feature we want, player_column is player_id,
# max_length is the most a player_id appears in the dataset
def create_player_sequences_fixed_length(data, target, player_column, max_length):
    """
    Create sequences of all past games for each player, then pad them to 'max_length'.
    """
    X_list, y_list = [], []

    # For every player_id, we group them together
    for p_id, group in data.groupby(player_column):
        # Drop player_id column to create player_features dataframe
        player_features = group.drop(columns=[player_column]).values
        # Retrive the points (target feature) for the player
        player_target = target[group.index].values

        # Create sequences of all past games, save the past features in X_list and corresponding
        # points (target feature) in Y_list
        for i in range(1, len(player_features)):
            seq = player_features[:i]
            X_list.append(seq)
            y_list.append(player_target[i])

    # Create X_padded 3D array with all sequences of uniform length (max_length)
    # 3D because for every player(1d) we want their previous features(2d) for every game(3d)
    num_features = X_list[0].shape[1] if X_list else 0
    X_padded = np.zeros((len(X_list), max_length, num_features), dtype=np.float32)

    # Create padded sequences
    for i, seq in enumerate(X_list):
        seq_len = len(seq)
        if seq_len <= max_length:
            X_padded[i, max_length - seq_len:, :] = seq
        else:
            X_padded[i, :, :] = seq[-max_length:]

    # Out: X(features), Y(variable to predict)
    return X_padded, np.array(y_list)

# ------------------------------------------------------------
# Helper Function: Build Model
# ------------------------------------------------------------
def build_lstm_model(input_shape):
    """
    Build an LSTM model with a Masking layer.
    """
    model = Sequential([
        Masking(mask_value=0.0, input_shape=input_shape),
        LSTM(64, activation='tanh', return_sequences=False),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

In [6]:
# ------------------------------------------------------------
# Fixed Period Train/Validation Split, Create sequences to prepare for model
# ------------------------------------------------------------
# Define training and validation periods
TRAIN_START_YEAR = 2015
TRAIN_END_YEAR = 2022
VALIDATION_YEAR = 2023

# Create masks for train and validation sets
train_mask = (df['game_year'] >= TRAIN_START_YEAR) & (df['game_year'] <= TRAIN_END_YEAR)
val_mask = (df['game_year'] == VALIDATION_YEAR)

train_data = df[train_mask]
val_data = df[val_mask]

print(f"Training data size: {len(train_data)} games")
print(f"Validation data size: {len(val_data)} games")

# Scale features (excluding player_id)
scaler = MinMaxScaler()
scaled_features_train = scaler.fit_transform(train_data[features].drop(columns=["player_id"]))
scaled_features_val = scaler.transform(val_data[features].drop(columns=["player_id"]))

# Create scaled dataframes with player_id
train_scaled = pd.DataFrame(scaled_features_train, 
                          index=train_data.index, 
                          columns=features[1:])
train_scaled["player_id"] = train_data["player_id"].values

val_scaled = pd.DataFrame(scaled_features_val, 
                        index=val_data.index, 
                        columns=features[1:])
val_scaled["player_id"] = val_data["player_id"].values

# Find maximum sequence length (most previous appearances of a player) across both datasets
def find_player_longest_sequence(data_df, id_col="player_id"):
    max_len = 0
    for _, group in data_df.groupby(id_col):
        length = len(group)
        max_len = max(max_len, length - 1)
    return max_len

# Call the find_player_longest_sequence() method
max_len_train = find_player_longest_sequence(train_scaled, "player_id")
max_len_val = find_player_longest_sequence(val_scaled, "player_id")
max_len_both = max(max_len_train, max_len_val)

print(f"Maximum sequence length: {max_len_both}")

# Create sequences for LSTM
X_train, y_train = create_player_sequences_fixed_length(
    train_scaled, train_data[target], "player_id", max_len_both
)
X_val, y_val = create_player_sequences_fixed_length(
    val_scaled, val_data[target], "player_id", max_len_both
)

Training data size: 187537 games
Validation data size: 14268 games
Maximum sequence length: 578


C:\Users\twans\anaconda3\envs\NBA\Lib\site-packages\keras\src\layers\core\masking.py:47: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/50
  32/5823 ━━━━━━━━━━━━━━━━━━━━ 19:45 205ms/step - loss: 170.5092 - mae: 9.7707

KeyboardInterrupt: 

In [ ]:
# ------------------------------------------------------------
# Build and Validate the model
# ------------------------------------------------------------
# Build and train the model, defines input shape and calls build_lstm_model helper
num_features = X_train.shape[2]
input_shape = (max_len_both, num_features)
model = build_lstm_model(input_shape=input_shape)
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Train and validate the the model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50, batch_size=32,
    verbose=1, callbacks=[early_stop]
)

# Final Results
y_pred = model.predict(X_val).flatten()
mae = mean_absolute_error(y_val, y_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print("\nValidation Results:")
print(f"Training Period: {TRAIN_START_YEAR}-{TRAIN_END_YEAR}")
print(f"Validation Year: {VALIDATION_YEAR}")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")